<a href="https://colab.research.google.com/github/anajrubim/Atividade_IHC/blob/main/ollamo_corrigido.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Steam Game Explorer com Ollama + Tool Calling
Consulta jogos da Steam usando o modelo `granite4:350m` com tool calling.

## 1. Instalação do Ollama

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


## 2. Dependências do sistema

In [ ]:
import sys

IS_COLAB = 'google.colab' in sys.modules

if IS_COLAB:
    print('Rodando no Colab. Instalando dependências...')
    !sudo apt-get update && sudo apt-get install -y zstd
    print('Dependências instaladas.')
else:
    print('Fora do Colab. Instale zstd manualmente se necessário.')

Rodando no Colab. Instalando dependências...
Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it 

## 3. Inicializar serviço Ollama (apenas uma vez)

In [ ]:
import threading
import subprocess
import time
import os

def run_ollama_serve():
    try:
        subprocess.Popen(
            ["ollama", "serve"],
            preexec_fn=os.setsid,
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL
        )
    except Exception as e:
        print(f"Erro ao iniciar o Ollama: {e}")

thread = threading.Thread(target=run_ollama_serve)
thread.start()

print("Aguardando o serviço Ollama iniciar...")
time.sleep(15)
print("Serviço Ollama iniciado.")

Aguardando o serviço Ollama iniciar...
Serviço Ollama iniciado.


## 4. Baixar modelo e instalar bibliotecas Python

In [ ]:
!ollama list
!ollama pull granite4:350m

NAME    ID    SIZE    MODIFIED 



In [ ]:
!uv pip install ollama rich pandas

Using Python 3.12.13 environment at: /usr
Checked 3 packages in 99ms


## 5. Coletar dados da Steam e salvar no banco

In [ ]:
import requests
import sqlite3

url = "https://store.steampowered.com/api/featuredcategories/"

resposta = requests.get(url)
dados = resposta.json()

conn = sqlite3.connect("steam.db")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS jogos (
    id INTEGER PRIMARY KEY,
    nome TEXT,
    preco_original INTEGER,
    preco_final INTEGER,
    desconto INTEGER,
    categoria TEXT
)
""")

CATEGORIAS_VALIDAS = {"specials", "top_sellers", "new_releases", "coming_soon", "recommended"}

for categoria, bloco in dados.items():

    if not isinstance(bloco, dict):
        continue
    nome_categoria = bloco.get("name", categoria)

    if str(nome_categoria).isdigit():
        continue

    items = bloco.get("items")

    if not items:
        continue

    for jogo in items:

        if "id" not in jogo:
            continue

        id_jogo = jogo.get("id")
        nome = jogo.get("name")

        preco_original = jogo.get("original_price")
        preco_final = jogo.get("final_price")
        desconto = jogo.get("discount_percent")

        if preco_original is None or preco_final is None:
            price = jogo.get("price_overview", {})
            preco_original = price.get("initial")
            preco_final = price.get("final")
            desconto = price.get("discount_percent")

        cursor.execute("""
        INSERT OR IGNORE INTO jogos
        (id, nome, preco_original, preco_final, desconto, categoria)
        VALUES (?, ?, ?, ?, ?, ?)
        """, (
            id_jogo,
            nome,
            preco_original,
            preco_final,
            desconto,
            nome_categoria
        ))

conn.commit()
conn.close()
print("Dados salvos com sucesso!")

Dados salvos com sucesso!


## 6. Funções de consulta

In [ ]:
import sqlite3
import pandas as pd

def carregar_dados():
    conn = sqlite3.connect("steam.db")
    df = pd.read_sql("SELECT * FROM jogos", conn)
    conn.close()
    return df

def tratar_precos(df):
    df = df.copy()
    if df["preco_original"].max() > 1000:  # ainda em centavos
        df["preco_original"] = df["preco_original"] / 100
        df["preco_final"] = df["preco_final"] / 100
    return df

def maiores_descontos(n=10, **kwargs):
    """Retorna os jogos com os maiores percentuais de desconto."""
    df = carregar_dados()
    df = tratar_precos(df)
    return df[df["desconto"] > 0].sort_values("desconto", ascending=False).head(n)

def jogos_baratos(preco=5, **kwargs):
    """Retorna os jogos com preço final abaixo do valor informado."""
    df = carregar_dados()
    df = tratar_precos(df)
    return df[df["preco_final"] < preco].sort_values("preco_final")

def media_por_categoria(**kwargs):
    """Retorna a média de preço final por categoria."""
    df = carregar_dados()
    df = tratar_precos(df)
    df_valido = df[df["preco_final"].notna() & (df["preco_final"] >= 0)]
    resultado = df_valido.groupby("categoria")["preco_final"].mean().round(2)
    return resultado

def jogos_gratis(**kwargs):
    """Retorna os jogos gratuitos (preço final igual a zero)."""
    df = carregar_dados()
    df = tratar_precos(df)
    return df[df["preco_final"] == 0]

def jogos_mais_caros(n=10, **kwargs):
    """Retorna os N jogos mais caros pelo preço final."""
    n_val = kwargs.get("n", n)
    try:
        n_val = int(n_val)
    except (ValueError, TypeError):
        n_val = n
    df = carregar_dados()
    df = tratar_precos(df)
    return df[df["preco_final"] > 0].sort_values("preco_final", ascending=False).head(n_val)

def busca_jogo(jogo, **kwargs):
    """Busca jogos pelo nome (parcial, sem distinção de maiúsculas)."""
    df = carregar_dados()
    df = tratar_precos(df)
    resultado = df[df["nome"].str.contains(jogo, case=False, na=False)]
    if resultado.empty:
        return "Nenhum jogo encontrado."
    return resultado[["id", "nome", "preco_original", "preco_final", "desconto", "categoria"]]

print("Funções carregadas com sucesso!")

Funções carregadas com sucesso!


## 7. Configuração do modelo e mapa de ferramentas

In [ ]:
import json
from rich import print
from ollama import chat

MODEL = 'granite4:350m'

tools_map = {
    "maiores_descontos": maiores_descontos,
    "jogos_baratos": jogos_baratos,
    "media_por_categoria": media_por_categoria,
    "busca_jogo": busca_jogo,
    "jogos_gratis": jogos_gratis,
    "jogos_mais_caros": jogos_mais_caros,
}

print("Modelo e ferramentas configurados!")

Modelo e ferramentas configurados!

## 8. Executor genérico de tool calls

In [ ]:
def executar_tool_call(response):
    """
    Recebe a resposta do modelo, identifica a ferramenta sugerida,
    filtra os argumentos válidos e executa a função.
    CORREÇÃO: lógica centralizada e correta para todas as funções.
    """
    if not response.message.tool_calls:
        print("[bold yellow]O modelo não sugeriu nenhuma ferramenta.[/bold yellow]")
        print(response.message.content)
        return None

    tool = response.message.tool_calls[0]
    nome_funcao = tool.function.name
    argumentos = tool.function.arguments or {}

    print(f"[bold green]Ferramenta sugerida:[/bold green] {nome_funcao}")
    print(f"[bold green]Argumentos recebidos:[/bold green] {argumentos}")

    if nome_funcao not in tools_map:
        print(f"[bold red]Ferramenta '{nome_funcao}' não encontrada no mapa.[/bold red]")
        return None

    func = tools_map[nome_funcao]
    import inspect
    params_validos = inspect.signature(func).parameters

    argumentos_filtrados = {}
    for k, v in argumentos.items():
        if k in params_validos and k not in ("kwargs",):
            param = params_validos[k]

            if param.annotation != inspect.Parameter.empty:
                try:
                    v = param.annotation(v)
                except (ValueError, TypeError):
                    pass
            argumentos_filtrados[k] = v

    result = func(**argumentos_filtrados)
    return result

print("Executor de tool calls configurado!")

Executor de tool calls configurado!

In [ ]:
import subprocess, threading, time, os, httpx

def ollama_esta_rodando():
    try:
        r = httpx.get("http://127.0.0.1:11434", timeout=3)
        return r.status_code == 200
    except:
        return False

def run():
    subprocess.Popen(
        ["ollama", "serve"],
        preexec_fn=os.setsid,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )

if ollama_esta_rodando():
    print("✅ Ollama já está rodando!")
else:
    print("⚠️ Iniciando Ollama...")
    threading.Thread(target=run).start()
    for i in range(40):
        time.sleep(1)
        if ollama_esta_rodando():
            print(f"✅ Pronto após {i+1}s!")
            break
        print(f"  aguardando... {i+1}s")
    else:
        print("❌ Falhou. Tente: Ambiente de execução → Reiniciar sessão")

✅ Ollama já está rodando!

### 9. Bot do telegram


In [ ]:
!pip install flask pyngrok openai-whisper gtts pydub --quiet
!apt-get install -y ffmpeg --quiet

import os
import threading
import tempfile
import traceback

import whisper
import requests as req
from gtts import gTTS
from flask import Flask, request, jsonify
from pyngrok import ngrok, conf

NGROK_TOKEN   = "3EvI69a55yTaYNOV278GDHJgAAp_5PCu5gwHRFHvqHjWcUSmn"
TELEGRAM_TOKEN = "8829614983:AAFOhnLoD5GqVSh3tSFPIxrOb0yKoeyAG20"

print("Carregando Whisper...")
whisper_model = whisper.load_model("base")
print("✅ Whisper carregado!")

conf.get_default().auth_token = NGROK_TOKEN
app = Flask(__name__)

def processar_pergunta(pergunta):
    from ollama import chat
    response = chat(
        model=MODEL,
        messages=[{"role": "user", "content": pergunta}],
        tools=list(tools_map.values())
    )
    result = executar_tool_call(response)

    if result is None:
        return "Não entendi. Tente perguntar sobre jogos baratos, descontos, jogos grátis, etc."

    if isinstance(result, str):
        return result

    linhas = []
    for _, row in result.iterrows():
        nome     = row.get("nome", "?")
        preco    = row.get("preco_final", None)
        desconto = row.get("desconto", 0)

        if preco == 0:
            preco_txt = "Grátis"
        elif preco is None:
            preco_txt = "Preço indisponível"
        else:
            preco_txt = f"R$ {preco:.2f}"

        if desconto and desconto > 0:
            linhas.append(f"{nome} — {preco_txt} ({int(desconto)}% de desconto)")
        else:
            linhas.append(f"{nome} — {preco_txt}")

    return "\n".join(linhas)


def transcrever_audio(file_path):
    result = whisper_model.transcribe(file_path, language="pt")
    return result["text"].strip()


def texto_para_audio(texto):
    tts = gTTS(text=texto, lang="pt", slow=False)
    tmp = tempfile.NamedTemporaryFile(suffix=".mp3", delete=False)
    tts.save(tmp.name)
    return tmp.name


def baixar_audio_telegram(file_id):
    resp      = req.get(f"https://api.telegram.org/bot{TELEGRAM_TOKEN}/getFile?file_id={file_id}")
    file_path = resp.json()["result"]["file_path"]
    audio     = req.get(f"https://api.telegram.org/file/bot{TELEGRAM_TOKEN}/{file_path}")
    tmp       = tempfile.NamedTemporaryFile(suffix=".ogg", delete=False)
    tmp.write(audio.content)
    tmp.close()
    return tmp.name


def enviar_texto(chat_id, texto):
    req.post(
        f"https://api.telegram.org/bot{TELEGRAM_TOKEN}/sendMessage",
        json={"chat_id": chat_id, "text": texto}
    )


def enviar_audio(chat_id, audio_path):
    with open(audio_path, "rb") as f:
        req.post(
            f"https://api.telegram.org/bot{TELEGRAM_TOKEN}/sendVoice",
            data={"chat_id": chat_id},
            files={"voice": f}
        )

MENSAGEM_AJUDA = (
    "Olá! Me manda uma mensagem ou áudio sobre jogos da Steam.\n\n"
    "Exemplos:\n"
    "• Jogos baratos\n"
    "• Maiores descontos\n"
    "• Jogos grátis\n"
    "• Jogos mais caros\n"
    "• Procure pelo jogo Baldur's Gate"
)

@app.route("/webhook", methods=["POST"])
def webhook():
    try:
        message = request.json.get("message", {})
        chat_id = message.get("chat", {}).get("id")

        if not chat_id:
            return jsonify({"ok": True})

        # Texto
        if "text" in message:
            texto = message["text"]
            if texto.startswith("/"):
                enviar_texto(chat_id, MENSAGEM_AJUDA)
            else:
                resposta    = processar_pergunta(texto)
                audio_path  = texto_para_audio(resposta)
                enviar_audio(chat_id, audio_path)
                os.unlink(audio_path)

        # Áudio
        elif "voice" in message or "audio" in message:
            file_id = message.get("voice", message.get("audio", {})).get("file_id")

            enviar_texto(chat_id, "🎙️ Transcrevendo seu áudio...")
            audio_path = baixar_audio_telegram(file_id)
            pergunta   = transcrever_audio(audio_path)
            os.unlink(audio_path)

            enviar_texto(chat_id, f"📝 Entendi: {pergunta}")
            resposta      = processar_pergunta(pergunta)
            audio_resposta = texto_para_audio(resposta)
            enviar_audio(chat_id, audio_resposta)
            os.unlink(audio_resposta)

    except Exception as e:
        print("❌ ERRO:", str(e))
        traceback.print_exc()

    return jsonify({"ok": True})

threading.Thread(target=lambda: app.run(port=5000), daemon=True).start()

url_publica  = ngrok.connect(5000).public_url
webhook_url  = f"{url_publica}/webhook"

req.post(
    f"https://api.telegram.org/bot{TELEGRAM_TOKEN}/setWebhook",
    json={"url": webhook_url}
)

print(f"🌐 URL pública: {webhook_url}")
print("✅ Bot rodando! Fale com ele no Telegram.")

Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 76 not upgraded.


Carregando Whisper...

✅ Whisper carregado!

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit


🌐 URL pública: https://headed-porridge-fascism.ngrok-free.dev/webhook

✅ Bot rodando! Fale com ele no Telegram.